In [ ]:
# 1. Select inputs. Ground truth is loaded only in the evaluation cell.
from pathlib import Path
from carth.pipeline.experiment import select_inputs

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "carth").is_dir())
TEST_SETS = {
    "demo": ROOT / "A3-Hackathon-Corpus/f00_demo",
    "corpus": ROOT / "A3-Hackathon-Corpus",
    "synthetic_dev": ROOT / "synthetic_testset/dev",
    "synthetic_holdout": ROOT / "synthetic_testset/holdout",
}
TEST_SET = "demo"
PATTERN = "**/*"
SUFFIXES = (".xlsx",)
LIMIT = 1  # None selects every matching input.

samples = select_inputs(TEST_SETS[TEST_SET], pattern=PATTERN, suffixes=SUFFIXES, limit=LIMIT)
[(sample.doc_id, str(sample.source.relative_to(ROOT))) for sample in samples]


# Extraction workbench

Run top to bottom. Change the selection above, inspect evidence, replace or compose the algorithms, then evaluate. Implementation lives in `carth/pipeline/`; this notebook contains configuration and assembly. The initial Excel mapping is a manual baseline for the demo table, not a general extractor. Reconfigure it when changing inputs. Keep holdout data for final evaluation, not tuning.


In [ ]:
# 2. Load and inspect source evidence once; reuse it across iterations.
%load_ext autoreload
%autoreload 2
from carth.pipeline.experiment import load_evidence, token_preview

evidence = load_evidence(samples)
selected_id = samples[0].doc_id
print(evidence[selected_id].get("sheets", evidence[selected_id].get("pages")))
token_preview(evidence[selected_id], limit=30)


In [ ]:
# 3. Assemble algorithms. Edit the Python modules and rerun from this cell.
from functools import partial
from carth.pipeline.algorithms import abstain, mapped_xlsx, TableSpec
from carth.pipeline.experiment import run_experiment

PREPROCESSORS = ()  # Source evidence -> source evidence; e.g. OCR or normalization.
ALGORITHMS = {"abstain": abstain}

# Explicit table mapping for the initial demo workbook. Disable for other layouts.
USE_TABLE_MAPPING = TEST_SET == "demo" and SUFFIXES == (".xlsx",) and len(samples) == 1
if USE_TABLE_MAPPING:
    table = TableSpec(
        sheet=evidence[selected_id]["sheets"][0]["sheet"],
        year=2025, unit="EUR", first_row=8,
        identity_column="A", name_column="B", level="asset",
        columns={
            "ahk_open":"C", "zugang":"D", "abgang":"E", "umbuchung":"F", "ahk_close":"G",
            "kum_afa_open":"H", "afa_gj":"I", "afa_abgang":"J", "zuschreibung":"K",
            "kum_afa_close":"L", "bw_close":"M", "bw_prev":"N",
        },
    )
    ALGORITHMS["mapped_xlsx"] = partial(mapped_xlsx, spec=table)

run = run_experiment(samples, ALGORITHMS, evidence=evidence,
                     preprocessors=PREPROCESSORS, output_root=ROOT / "out/experiments")
print(f"Saved predictions and run manifest: {run.directory}")
run.metrics


In [ ]:
# 4. Inspect the result and findings before opening ground truth.
algorithm = next(reversed(run.outputs))
result = run.outputs[algorithm][selected_id]
{"meta": result["meta"], "rows": result["rows"][:2], "findings": result["findings"]}


In [ ]:
# 5. Evaluate only the selected documents, after extraction.
from carth.pipeline.experiment import evaluate_run

evaluation = evaluate_run(run)
print("Inputs without answer keys:", evaluation["missing_keys"])
evaluation["summary"]


In [ ]:
# 6. Inspect score errors and compare algorithms. No extraction code receives these keys.
[{"algorithm": item["algorithm"], "doc_id": item["doc_id"], "score": item["score"],
  "right": item["right"], "wrong": item["wrong"], "blank": item["blank"], "extra": item["extra"],
  "errors": item["errors"]}
 for item in evaluation["documents"]]
